In [16]:
# Install first:
# pip install torch torch-geometric pandas scikit-learn

import torch
import pandas as pd
from torch_geometric.datasets import MovieLens100K
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import LightGCN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load MovieLens 100K
dataset = MovieLens100K(root="data/MovieLens100K")
data = dataset[0]

print(data)

# 2. Use user -> movie rating graph
edge_index = data["user", "rates", "movie"].edge_index

num_users = data["user"].num_nodes
num_movies = data["movie"].num_nodes
num_nodes = num_users + num_movies

# Convert movie node IDs so users and movies are in one graph
edge_index[1] += num_users

# Make graph undirected
edge_index = torch.cat(
    [edge_index, edge_index.flip(0)],
    dim=1
)

edge_index = edge_index.to(device)

# 3. Train/test split manually
num_edges = edge_index.size(1) // 2
perm = torch.randperm(num_edges)

train_size = int(0.8 * num_edges)
train_edges = edge_index[:, perm[:train_size]]
test_edges = edge_index[:, perm[train_size:]]

# Add reverse edges for training
train_edge_index = torch.cat(
    [train_edges, train_edges.flip(0)],
    dim=1
).to(device)

# 4. Build LightGCN model
model = LightGCN(
    num_nodes=num_nodes,
    embedding_dim=64,
    num_layers=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 5. Negative sampling function
def sample_negative_edges(pos_edges, num_users, num_movies):
    users = pos_edges[0]
    neg_movies = torch.randint(
        low=num_users,
        high=num_users + num_movies,
        size=(users.size(0),),
        device=device
    )
    return torch.stack([users, neg_movies], dim=0)

# 6. Training loop
epochs = 50

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    pos_edge_label_index = train_edges.to(device)
    neg_edge_label_index = sample_negative_edges(
        pos_edge_label_index,
        num_users,
        num_movies
    )

    pos_rank = model(train_edge_index, pos_edge_label_index)
    neg_rank = model(train_edge_index, neg_edge_label_index)

    loss = model.recommendation_loss(pos_rank, neg_rank)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss.item():.4f}")

# 7. Recommend top movies for one user
def recommend_movies(user_id, top_k=10):
    model.eval()

    user_node = torch.tensor([user_id], device=device)
    movie_nodes = torch.arange(num_users, num_users + num_movies, device=device)

    users = user_node.repeat(num_movies)

    edge_label_index = torch.stack([users, movie_nodes], dim=0)

    with torch.no_grad():
        scores = model(train_edge_index, edge_label_index)

    top_movies = scores.topk(top_k).indices
    recommended_movie_ids = top_movies.cpu().numpy()

    return recommended_movie_ids

for user_id in range(5):
    recommendations = recommend_movies(user_id, top_k=10)

    print(f"\nTop 10 movie recommendations for User {user_id}:")
    for movie_id in recommendations:
        print(f"Movie ID: {movie_id}")

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)
Epoch 010, Loss: 0.6014
Epoch 020, Loss: 0.4095
Epoch 030, Loss: 0.3869
Epoch 040, Loss: 0.3595
Epoch 050, Loss: 0.3326

Top 10 movie recommendations for User 0:
Movie ID: 49
Movie ID: 180
Movie ID: 99
Movie ID: 0
Movie ID: 173
Movie ID: 120
Movie ID: 55
Movie ID: 97
Movie ID: 126
Movie ID: 171

Top 10 movie recommendations for User 1:
Movie ID: 285
Movie ID: 299
Movie ID: 287
Movie ID: 257
Movie ID: 293
Movie ID: 327
Movie ID: 268
Movie ID: 312
Movie ID: 301
Movie ID: 747

Top 10 movie recommendations for User 2:
Movie ID: 285
Movie ID: 299
Movie ID: 287
Movie ID: 327
Movie ID: 312
Movie ID: 257
Movie ID: 325
Movie ID: 268
Movie ID: 293
Movie ID: 301

Top 10 movie recommendatio